# Gemma 4 31B Dense Oncology Text-Grounding Canary

This notebook is the dense Gemma 4 counterpart of the proven **non-deployable 512-record recovery canary**. It trains `unsloth/gemma-4-31B-it` with a language-only LoRA while the vision tower remains frozen. It preserves the original TRL 0.24 prompt/completion dataset, loss masking, training constants, and separate-conversation design for a controlled comparison with the 26B-A4B MoE variant.

The notebook explicitly disables thinking while formatting final-answer-only SFT records, audits the dense architecture and zero trainable vision tensors, and runs deterministic visual inference before training and after a cold adapter reload. Tool-call training remains intentionally excluded so architecture is the only experimental variable.

In [1]:
import gc
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, cast

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "garbage_collection_threshold:0.5,max_split_size_mb:256")
os.environ.setdefault("UNSLOTH_ENABLE_FLEX_ATTENTION", "0")
HF_HUB_CACHE = "/root/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = HF_HUB_CACHE

import unsloth
import torch
import transformers
import trl
from datasets import Dataset
from PIL import Image, ImageDraw

PROJECT_ROOT = Path("/workspace/training/pubmed")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/home/spark/projects/training/pubmed")

BASE_LLM = "unsloth/gemma-4-31B-it"
MODEL_VARIANT = "gemma4_31b_dense"
EXPECTED_MOE = False
EXPECTED_NUM_EXPERTS = None
EXPECTED_TOP_K_EXPERTS = None
MODEL_NAME_BASE = "pubmed_oncologist_gemma4_31b_dense_text_canary_512_sft"
INPUT_DATA_FILE = PROJECT_ROOT / "data/training-data/recovery/text_canary_512/train.jsonl"
INPUT_MANIFEST_FILE = INPUT_DATA_FILE.parent / "manifest.json"
OUTPUT_BASE_DIR = PROJECT_ROOT / "output/recovery" / MODEL_NAME_BASE
TRAIN_DIR = OUTPUT_BASE_DIR / "train"
LORA_OUTPUT_DIR = OUTPUT_BASE_DIR / "lora_adapters"

MAX_SEQ_LENGTH = 16384
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
SAVE_STEPS = 16
WARMUP_STEPS = 2
SEED = 3407
LORA_R = 32
LORA_ALPHA = 32

print(f"unsloth={unsloth.__version__} torch={torch.__version__} transformers={transformers.__version__} trl={trl.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")
print(f"Base: {BASE_LLM}")
print(f"Variant: {MODEL_VARIANT}")
print(f"Shared cache: {HF_HUB_CACHE}")
print(f"Input: {INPUT_DATA_FILE}")
print(f"Output: {OUTPUT_BASE_DIR}")
assert torch.cuda.is_available(), "CUDA is required"
assert trl.__version__ == "0.24.0", f"This notebook was verified against TRL 0.24.0, found {trl.__version__}"
assert INPUT_DATA_FILE.is_file(), f"Missing canary dataset: {INPUT_DATA_FILE}"
assert INPUT_MANIFEST_FILE.is_file(), f"Missing canary manifest: {INPUT_MANIFEST_FILE}"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth=2026.7.2 torch=2.10.0a0+b558c986e8.nv25.11 transformers=5.5.0 trl=0.24.0
GPU: NVIDIA GB10
Base: unsloth/gemma-4-31B-it
Variant: gemma4_31b_dense
Shared cache: /root/.cache/huggingface
Input: /workspace/training/pubmed/data/training-data/recovery/text_canary_512/train.jsonl
Output: /workspace/training/pubmed/output/recovery/pubmed_oncologist_gemma4_31b_dense_text_canary_512_sft


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

raw_rows = []
with INPUT_DATA_FILE.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        row = json.loads(line)
        prompt = row.get("prompt")
        completion = row.get("completion")
        assert isinstance(prompt, list) and [message.get("role") for message in prompt] == ["system", "user"], line_number
        assert isinstance(completion, list) and len(completion) == 1, line_number
        assert completion[0].get("role") == "assistant", line_number
        assert all(isinstance(message.get("content"), str) and message["content"].strip() for message in prompt + completion), line_number
        assert "<think>" not in completion[0]["content"].lower(), line_number
        assert "chat_template_kwargs" not in row, line_number
        row["chat_template_kwargs"] = {"enable_thinking": False}
        raw_rows.append(row)

manifest = json.loads(INPUT_MANIFEST_FILE.read_text(encoding="utf-8"))
input_sha256 = sha256_file(INPUT_DATA_FILE)
assert len(raw_rows) == 512, len(raw_rows)
assert manifest["rows"] == len(raw_rows)
assert manifest["output_sha256"] == input_sha256
assert manifest["tool_records_included"] is False
assert all(row["chat_template_kwargs"] == {"enable_thinking": False} for row in raw_rows)

record_counts = {}
for row in raw_rows:
    record_counts[row["record_type"]] = record_counts.get(row["record_type"], 0) + 1
print(f"Validated {len(raw_rows)} prompt/completion records")
print(f"SHA-256: {input_sha256}")
print(f"Record types: {record_counts}")
print("Gemma 4 thinking disabled explicitly for final-answer-only SFT")
print(json.dumps(raw_rows[0], indent=2)[:2000])

train_dataset = Dataset.from_list(raw_rows)

Validated 512 prompt/completion records
SHA-256: 0e8f5a4d0e0a2523283e380d450372ccd74bbba2d3cca306436cfcf87cdf3456
Record types: {'grounded_abstract': 480, 'missing_evidence': 16, 'general_replay': 16}
Gemma 4 thinking disabled explicitly for final-answer-only SFT
{
  "prompt": [
    {
      "role": "system",
      "content": "You are an oncology-focused medical language model for educational and research support.\n\nEvidence is conditional. Treat an abstract, tool result, or image analysis as available only when it is explicitly present in the conversation. Never invent a PMID, trial name, statistic, guideline version, measurement, retrieval result, or image finding. When evidence is missing, ask for it or state the limitation. Answer greetings and unrelated harmless questions normally without assuming a medical context. Do not provide a diagnosis or replace professional medical care."
    },
    {
      "role": "user",
      "content": "<evidence source=\"pubmed\" status=\"supplied\">

In [3]:
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    device_map=cast(Any, {"": 0}),
    cache_dir=HF_HUB_CACHE,
)
tokenizer = processor.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

assert model.config.model_type == "gemma4", model.config.model_type
text_config = model.config.text_config
actual_moe = bool(getattr(text_config, "enable_moe_block", False))
actual_num_experts = getattr(text_config, "num_experts", None)
actual_top_k_experts = getattr(text_config, "top_k_experts", None)
assert actual_moe is EXPECTED_MOE, (actual_moe, EXPECTED_MOE)
assert actual_num_experts == EXPECTED_NUM_EXPERTS, (actual_num_experts, EXPECTED_NUM_EXPERTS)
assert actual_top_k_experts == EXPECTED_TOP_K_EXPERTS, (actual_top_k_experts, EXPECTED_TOP_K_EXPERTS)

print(f"Loaded {BASE_LLM}")
print(f"Model: {type(model).__name__}")
print(f"Processor: {type(processor).__name__}")
print(f"Tokenizer: {type(tokenizer).__name__}, vocab={len(tokenizer)}")
print(f"pad_token={tokenizer.pad_token!r}, eos_token={tokenizer.eos_token!r}")
print(f"Architecture gate: moe={actual_moe}, experts={actual_num_experts}, active={actual_top_k_experts}")


def generate_multimodal(model_obj: Any, processor_obj: Any, messages: list[dict[str, Any]], max_new_tokens: int = 192) -> str:
    inputs = processor_obj.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    inputs = inputs.to(model_obj.device)
    input_length = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    return processor_obj.decode(output[0, input_length:], skip_special_tokens=True).strip()


def evaluate_visual_response(response: str) -> dict[str, bool]:
    normalized = response.casefold()
    return {
        "blue": "blue" in normalized,
        "red": "red" in normalized,
        "green": "green" in normalized,
        "ellipse_or_circle": any(term in normalized for term in ("ellipse", "oval", "circle")),
        "rectangle_or_square": any(term in normalized for term in ("rectangle", "rectangular", "square")),
        "vision_test_text": "vision test" in normalized,
    }


VISION_IMAGE = Image.new("RGB", (896, 896), color=(238, 238, 232))
draw = ImageDraw.Draw(VISION_IMAGE)
draw.rectangle((24, 24, 872, 872), outline=(10, 10, 10), width=12)
draw.text((330, 105), "VISION TEST", fill=(10, 10, 10))
draw.ellipse((95, 275, 355, 535), fill=(35, 90, 205))
draw.ellipse((541, 275, 801, 535), fill=(205, 45, 45))
draw.rectangle((335, 570, 561, 770), fill=(35, 155, 75))

VISION_MESSAGES = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": VISION_IMAGE},
            {
                "type": "text",
                "text": "This is a synthetic vision test card, not a medical image. Describe the visible text, colors, shapes, and their positions. Do not provide medical interpretation.",
            },
        ],
    }
]

FastVisionModel.for_inference(model)
base_visual_response = generate_multimodal(model, processor, VISION_MESSAGES)
base_visual_checks = evaluate_visual_response(base_visual_response)
assert base_visual_response, "Clean Gemma 4 base produced an empty visual response"
assert all(base_visual_checks.values()), {"response": base_visual_response, "checks": base_visual_checks}
print("CLEAN BASE VISUAL GATE PASSED")
print(json.dumps(base_visual_checks, indent=2))
print(base_visual_response)

Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: `flash_attention_2` is not supported for `gemma4` because max attention head dim 512 exceeds the Flash Attention 2 limit of 256 - defaulting to `sdpa`.


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

Loaded unsloth/gemma-4-31B-it
Model: Gemma4ForConditionalGeneration
Processor: Gemma4Processor
Tokenizer: GemmaTokenizer, vocab=262144
pad_token='<pad>', eos_token='<eos>'
Architecture gate: moe=False, experts=None, active=None
CLEAN BASE VISUAL GATE PASSED
{
  "blue": true,
  "red": true,
  "green": true,
  "ellipse_or_circle": true,
  "rectangle_or_square": true,
  "vision_test_text": true
}
This image is a synthetic vision test card featuring a light gray background enclosed by a thick black border.

The following elements are visible:
*   **Text:** The words "VISION TEST" are written in small, black, uppercase letters centered at the top of the image.
*   **Shapes and Colors:**
    *   A **blue circle** is positioned on the left side of the center.
    *   A **red circle** is positioned on the right side of the center.
    *   A **green square** is positioned centrally below the two circles.


In [4]:
model = FastVisionModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
    use_rslora=False,
    loftq_config=None,
)

trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
vision_markers = (
    "vision_tower",
    "vision_model",
    "vision_encoder",
    "visual",
    "embed_vision",
    "multi_modal_projector",
    "multimodal_projector",
    "mm_projector",
)
vision_trainable_names = [
    name for name in trainable_names
    if any(marker in name.casefold() for marker in vision_markers)
]
expert_trainable_names = [name for name in trainable_names if ".experts." in name.casefold()]
trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in model.parameters())

assert trainable_names, "LoRA attachment produced no trainable parameters"
assert not vision_trainable_names, f"Vision/projector parameters are trainable: {vision_trainable_names[:20]}"
assert all("lora_" in name for name in trainable_names), trainable_names[:20]
if EXPECTED_MOE:
    assert expert_trainable_names, "Gemma 4 MoE loaded, but Unsloth attached no LoRA tensors under expert modules"
else:
    assert not expert_trainable_names, f"Dense Gemma 4 unexpectedly has trainable expert tensors: {expert_trainable_names[:20]}"

print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
print(f"Trainable tensors: {len(trainable_names)}")
print(f"Expert LoRA tensors: {len(expert_trainable_names)}")
print(f"Vision/projector trainable tensors: {len(vision_trainable_names)}")
print("LORA TOPOLOGY GATE PASSED: architecture-aware language adapters attached with vision frozen.")

Trainable parameters: 244,858,880 / 16,874,581,552 (1.45%)
Trainable tensors: 820
Expert LoRA tensors: 0
Vision/projector trainable tensors: 0
LORA TOPOLOGY GATE PASSED: architecture-aware language adapters attached with vision frozen.


In [5]:
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer

assert all("image" not in row and "images" not in row for row in raw_rows), "This preservation canary must remain text-only"
FastVisionModel.for_training(model, use_gradient_checkpointing=True)

training_args = SFTConfig(
    output_dir=str(TRAIN_DIR),
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    completion_only_loss=True,
    dataset_num_proc=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=1,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
prepared_dataset = trainer.train_dataset
assert isinstance(prepared_dataset, Dataset), type(prepared_dataset)

print(f"Prepared rows: {len(prepared_dataset)}")
print(f"Prepared columns: {prepared_dataset.column_names}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Warmup steps: {WARMUP_STEPS}")
print("completion_only_loss=True, packing=False, text-only VLM preservation canary")

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=1):   0%|          | 0/512 [00:00<?, ? examples/s]

Prepared rows: 512
Prepared columns: ['input_ids', 'completion_mask', 'token_type_ids']
Effective batch: 8
Warmup steps: 2
completion_only_loss=True, packing=False, text-only VLM preservation canary


In [6]:
def contains_subsequence(sequence: list[int], subsequence: list[int]) -> bool:
    if not subsequence or len(subsequence) > len(sequence):
        return False
    width = len(subsequence)
    return any(sequence[index:index + width] == subsequence for index in range(len(sequence) - width + 1))

representative_indices = {}
for index, row in enumerate(raw_rows):
    representative_indices.setdefault(row["record_type"], index)

collate_batch = trainer.data_collator
assert callable(collate_batch), "Trainer did not configure a data collator"
mask_report = []
for record_type, index in sorted(representative_indices.items()):
    processed = prepared_dataset[index]
    batch = collate_batch([processed])
    labels = batch["labels"][0]
    input_ids = batch["input_ids"][0]
    trainable_mask = labels.ne(-100)
    assert trainable_mask.any(), f"No trainable labels for {record_type}"

    prompt_ids = tokenizer.apply_chat_template(
        raw_rows[index]["prompt"],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    assert labels[:len(prompt_ids)].eq(-100).all(), f"Prompt tokens are trainable for {record_type}"

    completion_mask = torch.tensor(processed["completion_mask"], dtype=torch.bool, device=input_ids.device)
    assert completion_mask.shape == input_ids.shape, f"Completion-mask length mismatch for {record_type}"
    assert torch.equal(trainable_mask, completion_mask), f"Trainer labels do not match the completion mask for {record_type}"
    trainable_ids = labels[trainable_mask].tolist()

    trainable_text = tokenizer.decode(trainable_ids, skip_special_tokens=False)
    assert "<|channel>thought" not in trainable_text, f"Thought channel leaked into labels for {record_type}"
    user_text = raw_rows[index]["prompt"][-1]["content"]
    user_prefix_ids = tokenizer.encode(user_text[:120], add_special_tokens=False)
    assert not contains_subsequence(trainable_ids, user_prefix_ids), f"User text leaked into labels for {record_type}"

    mask_report.append(
        {
            "record_type": record_type,
            "sequence_tokens": int(input_ids.ne(tokenizer.pad_token_id).sum()),
            "trainable_tokens": int(trainable_mask.sum()),
            "trainable_percent": round(100 * trainable_mask.float().mean().item(), 2),
            "decoded_label_preview": trainable_text[:300],
        }
    )

print(json.dumps(mask_report, indent=2))
print("LABEL MASK GATE PASSED: prompts and thought channels are masked; only final-answer completions are trainable.")

[
  {
    "record_type": "general_replay",
    "sequence_tokens": 131,
    "trainable_tokens": 6,
    "trainable_percent": 4.58,
    "decoded_label_preview": " you want summarized.<turn|>\n"
  },
  {
    "record_type": "grounded_abstract",
    "sequence_tokens": 370,
    "trainable_tokens": 32,
    "trainable_percent": 8.65,
    "decoded_label_preview": " EZH2 genes targeting RUNX3 affects proliferation, invasion, and metastasis of human colon cancer cells by Wnt/\u03b2-catenin signaling pathway.<turn|>\n"
  },
  {
    "record_type": "missing_evidence",
    "sequence_tokens": 156,
    "trainable_tokens": 26,
    "trainable_percent": 16.67,
    "decoded_label_preview": " have the study text or a tool result, so I cannot extract its design or response rates. Please supply the evidence.<turn|>\n"
  }
]
LABEL MASK GATE PASSED: prompts and thought channels are masked; only final-answer completions are trainable.


In [7]:
from transformers.trainer_utils import get_last_checkpoint

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
last_checkpoint = get_last_checkpoint(str(TRAIN_DIR))
if last_checkpoint:
    print(f"Resuming from {last_checkpoint}")
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting the canary from the clean base model")
    train_result = trainer.train()

print(f"Training complete: global_step={train_result.global_step}")
print(train_result.metrics)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Starting the canary from the clean base model


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 512 | Num Epochs = 1 | Total steps = 64
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 244,858,880 of 31,517,945,392 (0.78% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.154428
2,0.139173
3,0.016221
4,0.000003
5,0.067212
6,0.000464
7,0.008627
8,0.000569
9,0.000375
10,0.015411


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/recovery/pubmed_oncologist_gemma4_31b_dense_text_canary_512_sft/train/checkpoint-16/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/recovery/pubmed_oncologist_gemma4_31b_dense_text_canary_512_sft/train/checkpoint-32/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/recovery/pubmed_oncologist_gemma4_31b_dense_text_canary_512_sft/train/checkpoint-48/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/recovery/pubmed_oncologist_gemma4_31b_dense_text_canary_512_sft/train/checkpoint-64/tokenizer_config.json.


Training complete: global_step=64
{'train_runtime': 1294.9035, 'train_samples_per_second': 0.395, 'train_steps_per_second': 0.049, 'total_flos': 4.893572431041139e+16, 'train_loss': 0.020822959068407698, 'epoch': 1.0}


In [8]:
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(LORA_OUTPUT_DIR))
processor.save_pretrained(str(LORA_OUTPUT_DIR))

last_checkpoint = get_last_checkpoint(str(TRAIN_DIR))
complete = {
    "status": "trained_pending_behavioral_visual_tool_and_thinking_evaluation",
    "model_name": MODEL_NAME_BASE,
    "model_variant": MODEL_VARIANT,
    "base_model": BASE_LLM,
    "architecture": type(model).__name__,
    "model_type": model.config.model_type,
    "is_moe": actual_moe,
    "num_experts": actual_num_experts,
    "top_k_experts": actual_top_k_experts,
    "input_data_file": str(INPUT_DATA_FILE),
    "input_sha256": input_sha256,
    "input_manifest_file": str(INPUT_MANIFEST_FILE),
    "output_dir": str(OUTPUT_BASE_DIR),
    "lora_output_dir": str(LORA_OUTPUT_DIR),
    "last_checkpoint": last_checkpoint,
    "global_step": train_result.global_step,
    "num_train_epochs": NUM_EPOCHS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "load_in_4bit": True,
    "completion_only_loss": True,
    "packing": False,
    "enable_thinking_during_sft": False,
    "tool_records_included": False,
    "image_records_included": False,
    "finetune_vision_layers": False,
    "finetune_language_layers": True,
    "vision_projector_trainable_tensors": len(vision_trainable_names),
    "base_visual_checks": base_visual_checks,
    "base_visual_response": base_visual_response,
    "package_versions": {
        "unsloth": unsloth.__version__,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "trl": trl.__version__,
    },
    "completed_at": datetime.now(timezone.utc).isoformat(),
}
assert complete["vision_projector_trainable_tensors"] == 0
assert complete["is_moe"] is EXPECTED_MOE
assert complete["num_experts"] == EXPECTED_NUM_EXPERTS
assert complete["top_k_experts"] == EXPECTED_TOP_K_EXPERTS
(LORA_OUTPUT_DIR / "complete.json").write_text(json.dumps(complete, indent=2) + "\n", encoding="utf-8")
print(json.dumps(complete, indent=2))

Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/recovery/pubmed_oncologist_gemma4_31b_dense_text_canary_512_sft/lora_adapters/tokenizer_config.json.


{
  "status": "trained_pending_behavioral_visual_tool_and_thinking_evaluation",
  "model_name": "pubmed_oncologist_gemma4_31b_dense_text_canary_512_sft",
  "model_variant": "gemma4_31b_dense",
  "base_model": "unsloth/gemma-4-31B-it",
  "architecture": "PeftModelForCausalLM",
  "model_type": "gemma4",
  "is_moe": false,
  "num_experts": null,
  "top_k_experts": null,
  "input_data_file": "/workspace/training/pubmed/data/training-data/recovery/text_canary_512/train.jsonl",
  "input_sha256": "0e8f5a4d0e0a2523283e380d450372ccd74bbba2d3cca306436cfcf87cdf3456",
  "input_manifest_file": "/workspace/training/pubmed/data/training-data/recovery/text_canary_512/manifest.json",
  "output_dir": "/workspace/training/pubmed/output/recovery/pubmed_oncologist_gemma4_31b_dense_text_canary_512_sft",
  "lora_output_dir": "/workspace/training/pubmed/output/recovery/pubmed_oncologist_gemma4_31b_dense_text_canary_512_sft/lora_adapters",
  "last_checkpoint": "/workspace/training/pubmed/output/recovery/pubmed

In [9]:
FastVisionModel.for_inference(model)

SMOKE_PROMPTS = [
    "Hi",
    "Summarize the PubMed abstract I provided.",
    "What can you help me with?",
]
for user_prompt in SMOKE_PROMPTS:
    messages = [
        {"role": "system", "content": raw_rows[0]["prompt"][0]["content"]},
        {"role": "user", "content": user_prompt},
    ]
    response = generate_multimodal(model, processor, messages, max_new_tokens=160)
    assert response, f"Empty smoke response for: {user_prompt}"
    print(f"USER: {user_prompt}\nASSISTANT: {response}\n")

TypeError: string indices must be integers, not 'str'

In [ ]:
del trainer, model, processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

reloaded_model, reloaded_processor = FastVisionModel.from_pretrained(
    model_name=str(LORA_OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    device_map=cast(Any, {"": 0}),
    cache_dir=HF_HUB_CACHE,
)
reloaded_tokenizer = reloaded_processor.tokenizer
if reloaded_tokenizer.pad_token is None:
    reloaded_tokenizer.pad_token = reloaded_tokenizer.eos_token
reloaded_text_config = reloaded_model.config.text_config
assert reloaded_model.config.model_type == "gemma4", reloaded_model.config.model_type
assert bool(getattr(reloaded_text_config, "enable_moe_block", False)) is EXPECTED_MOE
assert getattr(reloaded_text_config, "num_experts", None) == EXPECTED_NUM_EXPERTS
assert getattr(reloaded_text_config, "top_k_experts", None) == EXPECTED_TOP_K_EXPERTS
FastVisionModel.for_inference(reloaded_model)

reload_messages = [
    {"role": "system", "content": raw_rows[0]["prompt"][0]["content"]},
    {"role": "user", "content": "Summarize the abstract I supplied."},
]
reload_response = generate_multimodal(
    reloaded_model,
    reloaded_processor,
    reload_messages,
    max_new_tokens=120,
)
assert reload_response, "Cold-reloaded Gemma 4 adapter produced an empty text response"
print(f"Cold text reload passed.\nASSISTANT: {reload_response}")

post_visual_response = generate_multimodal(
    reloaded_model,
    reloaded_processor,
    VISION_MESSAGES,
    max_new_tokens=192,
)
post_visual_checks = evaluate_visual_response(post_visual_response)
assert post_visual_response, "Cold-reloaded adapter produced an empty visual response"
assert all(post_visual_checks.values()), {
    "response": post_visual_response,
    "checks": post_visual_checks,
}
assert set(post_visual_checks) == set(base_visual_checks)
assert all(base_visual_checks.values())

visual_report = {
    "status": "passed",
    "model_variant": MODEL_VARIANT,
    "base_model": BASE_LLM,
    "model_type": reloaded_model.config.model_type,
    "is_moe": EXPECTED_MOE,
    "num_experts": EXPECTED_NUM_EXPERTS,
    "top_k_experts": EXPECTED_TOP_K_EXPERTS,
    "adapter": str(LORA_OUTPUT_DIR),
    "enable_thinking": False,
    "vision_trainable_tensors": len(vision_trainable_names),
    "base_checks": base_visual_checks,
    "post_training_checks": post_visual_checks,
    "base_response": base_visual_response,
    "post_training_response": post_visual_response,
    "evaluated_at": datetime.now(timezone.utc).isoformat(),
}
evaluation_dir = OUTPUT_BASE_DIR / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)
visual_report_path = evaluation_dir / "vision_preservation.json"
visual_report_path.write_text(json.dumps(visual_report, indent=2) + "\n", encoding="utf-8")

print("COLD-RELOAD GEMMA 4 ARCHITECTURE AND VISUAL PRESERVATION GATES PASSED")
print(json.dumps(post_visual_checks, indent=2))
print(post_visual_response)
print(f"Visual report: {visual_report_path}")